In [49]:
import numpy as np

In [50]:
MAX_PLAYERS = 18
MAX_CARD_ORDINALITY = 18
MAX_CARD_NUMBER = 18

In [51]:
from typing import List, Dict

import os
import torch

SINGLE_EXAMPLE_BYTES = 746

class MarkovValueDataset(torch.utils.data.Dataset):
    def __init__(self, data: List[Dict[str, np.ndarray]]):
        self.data = data

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

    def load_data(self, folder: str):
        # First, find all the files in the folder
        for file_name in os.listdir(folder):
            if file_name.endswith(".bin"):
                file_path = os.path.join(folder, file_name)
                try:
                    self.data.extend(self.try_load_batch(file_path))
                except ValueError as e:
                    print(f"Failed to load batch from {file_path}: {e}")

    def try_load_batch(self, file_path) -> List[Dict[str, np.ndarray]]:
        # Load a batch of data from a file
        batch_bytes = np.fromfile(file_path, dtype=np.uint8)
        num_bytes = len(batch_bytes)
        # If the number of bytes is not a multiple of the size of a single example, then we can't load it
        if num_bytes % SINGLE_EXAMPLE_BYTES != 0:
            raise ValueError(f"File {file_path} has an invalid number of bytes: {num_bytes}")
        num_examples = num_bytes // SINGLE_EXAMPLE_BYTES
        # Reshape the data into a 2D array
        batch_data = batch_bytes.reshape((num_examples, SINGLE_EXAMPLE_BYTES))
        target_bytes = batch_data[:, -8:]

        # Format the data
        targets = [np.frombuffer(b, dtype=np.float64) for b in target_bytes]
        for i in range(num_examples):
            if np.isnan(targets[i]).any():
                raise ValueError(f"Found NaN in target {i} of {file_path}")
            if np.isinf(targets[i]).any():
                print(target_bytes[i])
                raise ValueError(f"Found inf in target {i} of {file_path}")
        targets = np.array(targets)
        inputs = batch_data[:, :-8]
        # Change the input dtype to float64
        inputs = inputs.astype(np.float64)
        remaining_card_numbers = np.sum(inputs[:, -18:], axis=1)
        # Normalize the last 18 to be the proportion of the remaining card numbers
        inputs[:, -18:] = inputs[:, -18:] / remaining_card_numbers[:, None]
        # Normalize the data to be between -1 and 1, except for the last 18 columns
        inputs[:, :-18] = (inputs[:, :-18] * 2) - 1
        # Convert data to f32
        inputs = inputs.astype(np.float32)
        targets = targets.astype(np.float32)

        return [{"inputs": inputs[i], "target": targets[i]} for i in range(num_examples)]
    
    def split(self, frac: float) -> ('MarkovValueDataset', 'MarkovValueDataset'):
        # First, we need to shuffle the data
        np.random.shuffle(self.data)
        split_idx = int(len(self.data) * frac)
        return MarkovValueDataset(self.data[:split_idx]), MarkovValueDataset(self.data[split_idx:])
    
    @classmethod
    def from_file(cls, folder: str) -> 'MarkovValueDataset':
        dataset = cls([])
        dataset.load_data(folder)
        return dataset

In [52]:
class MarkovValueHeuristicModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = torch.nn.Sequential(
            torch.nn.Linear(SINGLE_EXAMPLE_BYTES - 8, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, 256),
            torch.nn.Dropout(0.1),
            torch.nn.ReLU(),
            torch.nn.Linear(256, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.model(x)

In [53]:
from tqdm.notebook import tqdm

epochs = 100
batch_size = 8
learning_rate = 1e-4

train_dataset, validation_dataset = MarkovValueDataset.from_file("data").split(0.8)
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_dataloader = torch.utils.data.DataLoader(validation_dataset, batch_size=batch_size, shuffle=True)

model = MarkovValueHeuristicModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-2)
lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=learning_rate, 
    epochs=epochs,
    steps_per_epoch=len(train_dataloader),
)

training_history = []
for epoch in range(epochs):
    total_training_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch + 1}")
    for idx, batch in enumerate(progress_bar):
        inputs = batch["inputs"]
        target = batch["target"]
        optimizer.zero_grad()
        output = model(inputs)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(output, target)
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_training_loss += loss.item()
        progress_bar.set_postfix({"avg loss": total_training_loss / (idx + 1)})

    total_validation_loss = 0
    with torch.no_grad():
        progress_bar = tqdm(validation_dataloader, desc=f"Validation {epoch + 1}")
        for idx, batch in enumerate(progress_bar):
            inputs = batch["inputs"]
            target = batch["target"]
            output = model(inputs)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(output, target)
            total_validation_loss += loss.item()
            progress_bar.set_postfix({"avg loss": total_validation_loss / (idx + 1)})

    training_history.append({
        "training_loss": total_training_loss / len(train_dataloader),
        "validation_loss": total_validation_loss / len(validation_dataloader)
    })

Epoch 1:   0%|          | 0/1485 [00:00<?, ?it/s]

Validation 1:   0%|          | 0/372 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/1485 [00:00<?, ?it/s]

Validation 2:   0%|          | 0/372 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/1485 [00:00<?, ?it/s]

...

Validation 98:   0%|          | 0/372 [00:00<?, ?it/s]

Epoch 99:   0%|          | 0/1485 [00:00<?, ?it/s]

Validation 99:   0%|          | 0/372 [00:00<?, ?it/s]

Epoch 100:   0%|          | 0/1485 [00:00<?, ?it/s]

Validation 100:   0%|          | 0/372 [00:00<?, ?it/s]

In [54]:
# Lets look at some example predictions
model.eval()
with torch.no_grad():
    for i in range(100):
        example = validation_dataset[i]
        inputs = torch.tensor(example["inputs"]).unsqueeze(0)
        target = example["target"]
        output = model(inputs)
        output = torch.sigmoid(output)
        print(f"Target: {target}, Output: {output.item()}")

Target: [0.10735], Output: 0.12225424498319626
Target: [0.2015], Output: 0.22079244256019592
Target: [0.23205], Output: 0.24198836088180542
Target: [0.06089333], Output: 0.0751194879412651
Target: [0.37415], Output: 0.37970224022865295
Target: [0.1243875], Output: 0.12769900262355804
Target: [0.19914], Output: 0.20854948461055756
Target: [0.06571333], Output: 0.07402481138706207
Target: [0.1502], Output: 0.14413626492023468
Target: [0.3145], Output: 0.3487672209739685
Target: [0.06465714], Output: 0.08471781760454178
Target: [0.08661819], Output: 0.10004815459251404
Target: [0.13878334], Output: 0.16110911965370178
Target: [0.133], Output: 0.08883339911699295
Target: [0.09663333], Output: 0.11265266686677933
Target: [0.05795625], Output: 0.07511139661073685
Target: [0.1195875], Output: 0.12905794382095337
Target: [0.1232875], Output: 0.12843717634677887
Target: [0.1722], Output: 0.18848730623722076
Target: [0.05846875], Output: 0.06905937194824219
Target: [0.06663334], Output: 0.081494